In [27]:
model = "llama3.2:1B"

#### Task 1: Create a Simple Chain for Summarization


**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.


**Hint: The five messages in LangChain:**


- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role


[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)

In [38]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Load the Ollama model
model="llama3.2:1b"
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Define the prompt template
summarization_prompt = ChatPromptTemplate([
    ("system", "Fasse den folgenden Text in genau einem Satz zusammen:"),
    ("human", "{text}"),
])

# ADD HERE YOUR CODE
# Create the LLMChain
summarization_chain = summarization_prompt | llm

# Sample text
text = """Over the last decade, deep learning has evolved massively to process and generate unstructured data like text, images, and video. 
These advanced AI models have gained popularity in various industries, and include large language models (LLMs). 
There is currently a significant level of fanfare in both the media and the industry surrounding AI,
and there’s a fair case to be made that Artificial Intelligence (AI), with these advancements,
is about to have a wide-ranging and major impact on businesses, societies, and individuals alike.
This is driven by numerous factors, including advancements in technology, high-profile applications, 
and the potential for transfor- mative impacts across multiple sectors."""

# ADD HERE YOUR CODE
# Invoke the chain
summary = summarization_chain.invoke({"text": text})
print(summary.content)

Deep learning hat in den letzten zehn Jahren massiv zu einem großen Fortschritt in der Umsetzung ungestregener Daten wie Text, Bilder und Videos geführt und ist nun sehr beliebt in verschiedenen Branchen.


In [29]:
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

print()

Hier sind 3 Sätze, die den Text zusammenfassen:

Deep learning hat in den letzten Decennien erhebliche Fortschritte gemacht, um un structurierte Daten wie Text, Bilder und Videos zu bearbeiten und zu generieren. Diese fortschrittlichen AI-Modelle haben sich in verschiedenen Branchen etabliert und umfassen großskalige Modellierungslösungen wie große Sprachmodelle (LLMs). Es ist ein breites Feld, das sich ausreichend Beweise für die enorme Auswirkung von künstlicher Intelligenz (KI) auf den Markt und Gesellschaften macht.


#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)

In [30]:
from langchain_core.tools import tool

@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multiply two integers and return their product."""
    return first_int * second_int


print(multiply.name)
print(multiply.description)
print(multiply.args)

# Werkzeug direkt testen
result = multiply.invoke({"first_int": 4, "second_int": 5})
print(result)

multiply
Multiply two integers and return their product.
{'first_int': {'title': 'First Int', 'type': 'integer'}, 'second_int': {'title': 'Second Int', 'type': 'integer'}}
20


In [31]:
# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Use bind_tools to pass the definition of our tool in as part of each call to the model, so that the model can invoke the tool
llm_with_tools = llm.bind_tools([multiply])

# When the model invokes the tool, this will show up in the AIMessage.tool_calls attribute of the output -> extract tool parameters from input text
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)

[{'name': 'multiply', 'args': {'first_int': '42', 'second_int': '5'}, 'id': 'd2a57855-c5f5-4004-b697-03a29e3b360c', 'type': 'tool_call'}]


In [32]:
# ADD HERE YOUR CODE
# Create the chain: pass the extracte tool parameters from the input text to the tool -> extract the arguments of the first tool_call
chain_with_tools = (
    llm_with_tools
    | (lambda msg: msg.tool_calls[0]["args"])
    | multiply
)

# Run chain
result=chain_with_tools.invoke("whats 5 times forty two")
print(result)

210


#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:** 

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define the system prompt.
- Define tools.
- Create an Agent using the Ollama model, system-prompt and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.


**Useful links:**

- [How To](https://docs.langchain.com/oss/javascript/langchain/agents)

In [33]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

# Load the Ollama model
model="llama3.2:3b"
llm = ChatOllama(model=model, temperature=0)

# ADD HERE YOUR CODE
agent_prompt = (
    "You are a math assistant. "
    "Always use the provided tools for calculations. "
    "For multi-step tasks, use the previous tool result "
    "as input for the next calculation. "
    "When finished, give the final answer in German."
)


# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int


@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent


tools = [add, exponentiate]

In [34]:
# ADD HERE YOUR CODE
# Construct the agent
agent_with_tools = create_agent(
    model=llm,
    tools=tools,
    system_prompt=agent_prompt
)

In [35]:
response = agent_with_tools.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Call exponentiate with base=3 and exponent=5. "
                    "Then call add using first_int=the previous result "
                    "and second_int=12. "
                    "Use exactly these argument names. "
                    "After both tools succeed, give the final answer."
                )
            }
        ]
    },
    config={"recursion_limit": 20}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Call exponentiate with base=3 and exponent=5. Then call add using first_int=the previous result and second_int=12. Use exactly these argument names. After both tools succeed, give the final answer.
================================== Ai Message ==================================
Tool Calls:
  exponentiate (25757ace-9060-4e59-9d30-a3a987645043)
 Call ID: 25757ace-9060-4e59-9d30-a3a987645043
  Args:
    base: 3
    exponent: 5
================================= Tool Message =================================
Name: exponentiate

243
================================== Ai Message ==================================
Tool Calls:
  add (ec666e3d-fa54-4f10-9590-d65a217b33ba)
 Call ID: ec666e3d-fa54-4f10-9590-d65a217b33ba
  Args:
    first_int: 243
    second_int: 12
================================= Tool Message =================================
Name: add

255
================================== Ai Message ============

In [37]:
print(llm.model)

llama3.2:3b
